# 🧠 MindMap: Machine Learning Core Pipeline
### Focused on Feature Extraction, Model Training, and Serialization

This notebook implements the core research methodology for the MindMap project. 
**Note:** Ensure 'data/student_data.csv' exists before running this. If not, run 'python generate_data_and_train.py'.

In [5]:
import pandas as pd
import numpy as np
import json
import joblib
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from datetime import datetime
from xgboost import XGBClassifier

# Constants and Directory Setup
os.makedirs("models", exist_ok=True)
CAT_COLS = ["gender", "student_type", "education_level", "branch",
            "sleep_racing", "friday_mood", "self_doubt", "goal_setting_freq",
            "financial_stress", "living_situation"]
NUM_COLS = ["study_hours", "attend_hours", "confidence", "prefers_online_video",
            "active_learner", "sleep_hours", "exam_blank", "overwhelm",
            "workload_freq", "time_enough", "puzzle_score", "engagement",
            "phone_hours", "friends_count", "sgpa", "atkt", "has_internship",
            "has_job", "in_club", "edtech_platforms", "hobbies_count",
            "social_interaction_freq"]

## 1. Data Ingestion & Preprocessing
Loading the simulated dataset and performing feature engineering.

In [6]:
import os

if not os.path.exists("data/student_data.csv"):
    print("❌ Error: 'data/student_data.csv' not found!")
    print("💡 Solution: Run 'python generate_data_and_train.py' in your terminal first.")
else:
    # Load existing data
    df = pd.read_csv("data/student_data.csv")
    print("✅ Data loaded successfully. Shape:", df.shape)
    
    # Label encoding for categorical variables
    le_dict = {}
    df_enc = df.copy()
    for col in CAT_COLS:
        le = LabelEncoder()
        df_enc[col + "_enc"] = le.fit_transform(df_enc[col].astype(str))
        le_dict[col] = le

    # Mapping ordinal features
    sleep_racing_map = {"Never": 0, "Sometimes": 1, "Often": 2, "Always": 3}
    goal_map = {"Never": 0, "Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
    df_enc["sleep_racing_num"] = df_enc["sleep_racing"].map(sleep_racing_map)
    df_enc["goal_freq_num"] = df_enc["goal_setting_freq"].map(goal_map)

    FEATURE_COLS = NUM_COLS + [c + "_enc" for c in CAT_COLS] + ["sleep_racing_num", "goal_freq_num"]
    X = df_enc[FEATURE_COLS].fillna(0)

    # Standardization
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Target Encoding
    le_mode = LabelEncoder()
    le_stress = LabelEncoder()
    y_mode = le_mode.fit_transform(df["learning_mode"])
    y_stress = le_stress.fit_transform(df["stress_level"])
    y_risk = df["at_risk_flag"].values

    print("✅ Data processing complete. Numerical vector size:", X_scaled.shape)

❌ Error: 'data/student_data.csv' not found!
💡 Solution: Run 'python generate_data_and_train.py' in your terminal first.


## 2. Experimental Execution (Model Training)
Training the core classification and clustering models.

In [4]:
if 'X_scaled' in locals():
    X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X_scaled, y_mode, test_size=0.2, random_state=42)
    X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_scaled, y_stress, test_size=0.2, random_state=42)
    X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_scaled, y_risk, test_size=0.2, random_state=42)

    # 2.1 Learning Mode (Random Forest)
    rf_mode = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight="balanced")
    rf_mode.fit(X_tr_m, y_tr_m)
    print(f"Learning Mode RF Accuracy: {rf_mode.score(X_te_m, y_te_m):.4f}")

    # 2.2 Stress Level (XGBoost)
    xgb_stress = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42)
    xgb_stress.fit(X_tr_s, y_tr_s)
    print(f"Stress Level XGB Accuracy: {xgb_stress.score(X_te_s, y_te_s):.4f}")

    # 2.3 At-Risk Identification (Logistic Regression)
    lr_risk = LogisticRegression(class_weight="balanced", random_state=42)
    lr_risk.fit(X_tr_r, y_tr_r)
    print(f"At-Risk LR Accuracy: {lr_risk.score(X_te_r, y_te_r):.4f}")

    # 2.4 Behavioral Profiling (K-Means)
    km = KMeans(n_clusters=6, random_state=42, n_init=10)
    km.fit(X_scaled)
    print("K-Means Clustering complete.")
else:
    print("⚠️ Skipping training because data was not loaded.")

⚠️ Skipping training because data was not loaded.


## 3. Model Serialization (Persistence)
Exporting artifacts for system deployment.

In [ ]:
if 'rf_mode' in locals():
    joblib.dump(rf_mode, "models/model_learning_mode.pkl")
    joblib.dump(xgb_stress, "models/model_stress.pkl")
    joblib.dump(lr_risk, "models/model_risk.pkl")
    joblib.dump(km, "models/kmeans_archetype.pkl")
    joblib.dump(scaler, "models/scaler.pkl")
    joblib.dump(le_dict, "models/label_encoders.pkl")
    joblib.dump(le_mode, "models/le_mode.pkl")
    joblib.dump(le_stress, "models/le_stress.pkl")
    joblib.dump(FEATURE_COLS, "models/feature_cols.pkl")

    print("✅ All models serialized to /models/ directory.")
else:
    print("⚠️ Models were not trained. Nothing to save.")